In [ ]:
!pip install -q numpy scikit-learn matplotlib requests python-dotenv

In [ ]:
from google.colab import userdata
import requests

In [ ]:
import math
import numpy as np
import matplotlib.pyplot as plt

from sklearn.decomposition import PCA

In [ ]:
API_KEY = userdata.get("OPENROUTER_API_KEY")

if not API_KEY:
    raise ValueError(
        "OPENROUTER_API_KEY was not found. "
        "Add it to Colab's secrets (key icon in the sidebar) before continuing."
    )

In [ ]:
response = requests.post(
    "https://openrouter.ai/api/v1/embeddings",
    headers={
        "Authorization": f"Bearer {API_KEY}",
        "Content-Type": "application/json",
    },
    json={
        "model": "openai/text-embedding-3-small",
        "input": "The quick brown fox jumps over the lazy dog",
    },
)

# Fail fast with a clear error instead of a confusing KeyError later
# if the API call itself did not succeed.
response.raise_for_status()

In [ ]:
data = response.json()
embedding = data["data"][0]["embedding"]
print(f"Embedding dimension: {len(embedding)}")

In [ ]:
data

### 1. Euclidean Distance

In [ ]:
def euclidean_distance(embedding_a: list[float], embedding_b: list[float]) -> float:
    """Compute the Euclidean distance between two embedding vectors.

    Args:
        embedding_a: First embedding vector.
        embedding_b: Second embedding vector.

    Returns:
        The Euclidean distance between the two vectors.

    Raises:
        ValueError: If the vectors have different dimensions.
    """
    if len(embedding_a) != len(embedding_b):
        raise ValueError("Embeddings must have the same dimension.")

    squared_diff_sum = sum((a - b) ** 2 for a, b in zip(embedding_a, embedding_b))

    return math.sqrt(squared_diff_sum)

### 2. Cosine Distance

In [ ]:
def cosine_distance(embedding_a: list[float], embedding_b: list[float]) -> float:
    """Compute the cosine distance (1 - cosine similarity) between two vectors.

    Args:
        embedding_a: First embedding vector.
        embedding_b: Second embedding vector.

    Returns:
        The cosine distance between the two vectors (0 = identical direction,
        1 = orthogonal, 2 = opposite direction).

    Raises:
        ValueError: If the vectors have different dimensions, or if either
            vector has zero magnitude (cosine similarity is undefined for a
            null vector).
    """
    if len(embedding_a) != len(embedding_b):
        raise ValueError("Embeddings must have the same dimension.")

    dot_product = sum(a * b for a, b in zip(embedding_a, embedding_b))
    norm_a = math.sqrt(sum(a ** 2 for a in embedding_a))
    norm_b = math.sqrt(sum(b ** 2 for b in embedding_b))

    if norm_a == 0 or norm_b == 0:
        raise ValueError("Cannot compute cosine distance with a null vector.")

    cosine_similarity = dot_product / (norm_a * norm_b)

    return 1 - cosine_similarity

### 3. Testing the functions

In [ ]:
embedding_a = [1, 0, 0]
embedding_b = [0, 1, 0]
embedding_c = [1, 0, 0]

print("A -> B")
print("Euclidean:", euclidean_distance(embedding_a, embedding_b))
print("Cosine:", cosine_distance(embedding_a, embedding_b))

print("\nA -> C")
print("Euclidean:", euclidean_distance(embedding_a, embedding_c))
print("Cosine:", cosine_distance(embedding_a, embedding_c))

print("\nB -> C")
print("Euclidean:", euclidean_distance(embedding_b, embedding_c))
print("Cosine:", cosine_distance(embedding_b, embedding_c))

### 4. Applying to the terms from class

In [ ]:
embedding_cat = [1.0, 0.9, 0.1]
embedding_feline = [0.9, 1.0, 0.1]
embedding_dog = [0.8, 0.7, 0.2]

embedding_car = [0.1, 0.2, 1.0]
embedding_truck = [0.1, 0.3, 0.9]
embedding_motorcycle = [0.2, 0.3, 0.8]

embedding_banana = [0.7, 0.1, 0.2]
embedding_apple = [0.6, 0.2, 0.2]
embedding_guava = [0.5, 0.2, 0.3]

In [ ]:
print("Cat x Feline")
print("Euclidean:", euclidean_distance(embedding_cat, embedding_feline))
print("Cosine:", cosine_distance(embedding_cat, embedding_feline))

print("\nCar x Truck")
print("Euclidean:", euclidean_distance(embedding_car, embedding_truck))
print("Cosine:", cosine_distance(embedding_car, embedding_truck))

print("\nBanana x Apple")
print("Euclidean:", euclidean_distance(embedding_banana, embedding_apple))
print("Cosine:", cosine_distance(embedding_banana, embedding_apple))

### 5. Real Embeddings and 3D Visualization
The previous section used hand-picked vectors to illustrate the metrics.
Here we fetch **real embeddings** from the API for a set of related terms, reduce them from 1536 dimensions to 3 with PCA, and plot them in 3D to see how semantically related terms cluster together.

In [ ]:
def fetch_embeddings(texts: list[str]) -> list[list[float]]:
    """Fetch embeddings for a batch of texts from the OpenRouter API.

    Args:
        texts: List of strings to embed.

    Returns:
        A list of embedding vectors, in the same order as `texts`.

    Raises:
        requests.HTTPError: If the API request fails.
    """
    response = requests.post(
        "https://openrouter.ai/api/v1/embeddings",
        headers={
            "Authorization": f"Bearer {API_KEY}",
            "Content-Type": "application/json",
        },
        json={
            "model": "openai/text-embedding-3-small",
            "input": texts,
        },
    )
    response.raise_for_status()

    payload = response.json()
    # The API preserves input order via the "index" field, so we sort on it
    # defensively instead of assuming the response order matches the request.
    sorted_items = sorted(payload["data"], key=lambda item: item["index"])

    return [item["embedding"] for item in sorted_items]

In [ ]:
terms_by_category = {
    "animal": ["cat", "feline", "dog"],
    "vehicle": ["car", "truck", "motorcycle"],
    "fruit": ["banana", "apple", "guava"],
}

terms = [term for group in terms_by_category.values() for term in group]
categories = [
    category
    for category, group in terms_by_category.items()
    for _ in group
]

In [ ]:
real_embeddings = fetch_embeddings(terms)
embeddings_matrix = np.array(real_embeddings)

print(f"Fetched {embeddings_matrix.shape[0]} embeddings "
      f"of dimension {embeddings_matrix.shape[1]}")

In [ ]:
pca = PCA(n_components=3, random_state=42)
embeddings_3d = pca.fit_transform(embeddings_matrix)

explained_variance = pca.explained_variance_ratio_.sum()
print(f"Variance explained by the first 3 components: {explained_variance:.1%}")

In [ ]:
category_colors = {
    "animal": "tab:orange",
    "vehicle": "tab:blue",
    "fruit": "tab:green",
}

fig = plt.figure(figsize=(9, 7))
ax = fig.add_subplot(111, projection="3d")

for term, category, (x, y, z) in zip(terms, categories, embeddings_3d):
    ax.scatter(x, y, z, color=category_colors[category], s=60)
    ax.text(x, y, z, term, fontsize=9)

# One legend entry per category instead of one per point
legend_handles = [
    plt.Line2D([0], [0], marker="o", linestyle="", color=color, label=category)
    for category, color in category_colors.items()
]
ax.legend(handles=legend_handles, title="Category")

ax.set_xlabel("PC 1")
ax.set_ylabel("PC 2")
ax.set_zlabel("PC 3")
ax.set_title("Real Embeddings Reduced to 3D (PCA)")

plt.tight_layout()
plt.show()